# **REINVENT4를 이용한 de novo 분자 설계**
이 실습을 마치면 다음을 할 수 있습니다:

*  분자 생성을 위한 강화학습(RL)의 기본 원리를 이해한다
*  REINVENT4가 RL로 분자 물성을 최적화하는 방식을 설명한다
*  다양한 분자 설계 과제에 맞게 REINVENT4를 설정한다
*  sigma와 diversity filter가 학습에 미치는 영향을 분석한다
*  sampling·scoring 워크플로우로 생성 분자를 평가한다
*  생성 분자의 다양성을 IntDiv·#Circles로 측정한다
*  전이학습(transfer learning)으로 확장하는 방법을 안다 *(마지막 선택 과제)*

> **런타임 선택** — 이 노트북은 **CPU 런타임에서 그대로 동작**합니다. 시간이 오래 걸리는
> 학습 셀은 모두 선택 사항이고, 미리 학습해 둔 결과를 자동으로 받아 쓰도록 되어 있습니다.
> GPU 런타임(`런타임 > 런타임 유형 변경 > T4 GPU`)을 쓰면 학습 셀을 직접 돌려볼 수 있습니다.
> 아래 설정 확인 셀이 GPU를 자동으로 감지해 `DEVICE`에 넣어 주므로, 그 외에 바꿀 것은 없습니다.


먼저 REINVENT4를 내려받고 필요한 패키지를 설치합니다. 설치가 끝나면 Colab이 세션을 재시작해야 하므로, 아래 셀을 계속 진행하기 전에 재시작(restart)을 확인해 주세요. 설치에는 약 5분 정도 걸립니다.

In [ ]:
# ============================================================
#  REINVENT4 내려받기 + 설치
# ============================================================
import os

# ── 1) 코드·prior 내려받기 — 이미 있으면 건너뜁니다 (다시 실행해도 안전) ──
#  강의 저장소 Releases 의 경량 배포본 (약 320 MB).
#  실습에 쓰는 prior·체크포인트만 담겨 있어 원본보다 훨씬 빨리 받습니다.
CODE_URL = ("https://github.com/eunjoolee122/2026-aidrugdiscovery"
            "/releases/download/asset-v3/reinvent4_slim.zip")

if not os.path.exists("/content/REINVENT4"):
    !wget -q --show-progress -O /content/reinvent4.zip $CODE_URL
    !unzip -q /content/reinvent4.zip -d /content/

assert os.path.isdir("/content/REINVENT4/reinvent"), \
    "내려받기에 실패했습니다 — CODE_URL 주소와 저장소 공개 여부를 확인하세요"

%cd /content/REINVENT4

# ── 2) 의존 패키지 ────────────────────────────────────────────────
#  저장소의 reinvent_requirements.txt 는 Python 3.10 시절 버전으로 고정돼 있어
#  최신 Colab 런타임에서는 pillow·numpy 등이 소스 빌드에 실패합니다.
#  아래는 버전을 고정하지 않고 현재 런타임에 맞춰 설치하는 조합입니다.
%pip install -q "mmpdb>=2.1,<3" tomli tenacity xxhash apted molvs funcy \
                python-dotenv pyyaml requests tqdm pydantic matplotlib \
                rdkit tensorboard "numpy<3" "pandas<3" pillow mols2grid

#  torch·torchvision은 Colab에 이미 들어 있습니다 (GPU 런타임이면 CUDA 빌드).
#  없다는 오류가 나면 아래 줄의 주석을 푸세요 — CPU 런타임 기준입니다.
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

# ── 3) reinvent 패키지 등록 (의존성 재해석 없이) ──────────────────
%pip install -q --no-deps /content/REINVENT4


설치가 끝나면 아래 셀로 확인합니다. `REINVENT 4.6.22 ...` 가 찍히면 정상입니다.
(설치 직후 Colab이 세션 재시작을 요구하면, 재시작한 뒤 이 셀부터 이어서 실행하세요.)

In [ ]:
import os, sys, numpy as np, pandas as pd, rdkit, torch
os.chdir("/content/REINVENT4")

# 이 노트북의 모든 TOML 설정이 이 값을 씁니다.
# GPU 런타임이면 "cuda", 아니면 "cpu" 가 자동으로 들어갑니다.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("python  ", os.sys.version.split()[0])
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("rdkit   ", rdkit.__version__)
print("torch   ", torch.__version__)
print("device  ", DEVICE)
if DEVICE == "cuda":
    print("         ", torch.cuda.get_device_name(0))
else:
    print("          GPU로 바꾸려면 런타임 > 런타임 유형 변경 > T4 GPU 후 이 셀부터 다시 실행")
print("-" * 40)
!{sys.executable} -m reinvent.Reinvent --version


---
### 📁 산출물 경로

실습을 돌리면 TOML·로그·체크포인트·결과 CSV가 쌓입니다. 아이템별로 폴더를 나눠 두면
어떤 파일이 어느 실습에서 나온 것인지 헷갈리지 않습니다.

```
/content/REINVENT4/work/
├── 01_sigma/      sigma 비교
├── 02_rotbond/    회전결합 reward
└── 03_trxr/       TrxR 물성 프로파일 과제
```

아래 셀을 한 번 실행해 두면, 이후 모든 셀이 이 경로를 씁니다.

In [ ]:
# ============================================================
#  경로 체계 — 실습 아이템별로 산출물을 분리한다
# ============================================================
import os

ROOT = "/content/REINVENT4"
WORK = os.path.join(ROOT, "work")

ITEMS = {"sigma":   "01_sigma",     # 3절  sigma 비교
         "rotbond": "02_rotbond",   # 4절  회전결합 reward
         "trxr":    "03_trxr",     # 5절  TrxR 과제
         "libinvent": "04_libinvent"}  # 6절  LibInvent R-group


def wdir(item):
    """아이템 폴더 경로 (없으면 만든다)"""
    p = os.path.join(WORK, ITEMS[item])
    os.makedirs(p, exist_ok=True)
    return p


def wfile(item, name):
    """아이템 폴더 안의 파일 경로"""
    return os.path.join(wdir(item), name)


for k in ITEMS:
    wdir(k)

print("산출물 경로")
for k, v in ITEMS.items():
    print(f"  {k:8s} → {os.path.join(WORK, v)}")


In [ ]:
# 분석·시각화용 import (설치는 위 셀에서 이미 끝났습니다)
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import mols2grid


---
### 🛟 보충 자료 자동 내려받기 (안전장치)

이 실습에는 시간이 오래 걸리는 강화학습 셀이 몇 군데 있습니다. 학습을 건너뛰거나 도중에
세션이 끊겨도 **뒤 단계를 이어서 진행할 수 있도록**, 필요한 파일이 없으면 미리 학습해 둔
파일을 자동으로 내려받습니다.

아래 셀을 한 번 실행해 두세요. 이미 파일이 있으면 아무것도 하지 않습니다.

In [ ]:
# ============================================================
#  보충 자료 자동 내려받기 (학습을 건너뛰거나 실패했을 때의 안전장치)
# ============================================================
ASSETS_URL = ("https://github.com/eunjoolee122/2026-aidrugdiscovery/releases/download/asset-v3/assets.zip")

import os, shutil, subprocess

_ZIP = "/content/assets.zip"


def _locate(name):
    """푼 결과에서 이름이 맞는 것을 깊이 상관없이 찾는다.
       zip이 content/assets/ 같은 겹을 달고 있어도 걸린다."""
    direct = os.path.join(ROOT, name)
    if os.path.exists(direct):
        return direct
    for cur, dirs, files in os.walk(ROOT):
        if os.path.basename(cur) == "work":
            dirs[:] = []                       # 이미 정리된 아이템 폴더는 건너뛴다
            continue
        for cand in dirs + files:
            if cand == name:
                return os.path.join(cur, cand)
    return None


def ensure(item, *names):
    """없는 파일만 보충 자료 zip에서 꺼내 해당 아이템 폴더에 놓는다."""
    dests = {n: wfile(item, n) for n in names}
    missing = [n for n in names if not os.path.exists(dests[n])]

    if not missing:
        print("[OK]   이미 있음  -  " + ", ".join(names))
        return [dests[n] for n in names]

    print("[..]   없는 파일  -  " + ", ".join(missing))

    if not os.path.exists(_ZIP):
        print("[..]   보충 자료를 내려받는 중 ...")
        subprocess.run(["wget", "-q", "-O", _ZIP, ASSETS_URL])
        if not os.path.exists(_ZIP) or os.path.getsize(_ZIP) < 1024:
            if os.path.exists(_ZIP):
                os.remove(_ZIP)
            raise RuntimeError(
                "내려받기에 실패했습니다: " + ASSETS_URL
                + "\n  · 주소와 저장소 공개 여부를 확인하세요."
                + "\n  · 또는 앞의 학습 셀을 직접 실행해 파일을 만드세요."
            )

    subprocess.run(["unzip", "-n", "-q", _ZIP, "-d", ROOT], check=True)

    for n in missing:
        staged = _locate(n)
        if staged and not os.path.exists(dests[n]):
            shutil.move(staged, dests[n])

    still = [n for n in missing if not os.path.exists(dests[n])]
    if still:
        raise RuntimeError(
            "보충 자료 zip에 다음이 없습니다: " + ", ".join(still)
            + "\n  · zip 안에 그 이름이 실제로 있는지: !unzip -l " + _ZIP
        )

    print("[OK]   준비 완료  -  " + ", ".join(missing))
    return [dests[n] for n in names]


print("안전장치 준비 완료. 이제 아래 셀들을 순서대로 실행하세요.")

---
### ⚙️ REINVENT 실행 헬퍼

아래 셀들은 `!python` 대신 **이 함수로** REINVENT를 실행합니다.
`!python` 은 노트북 커널과 다른 파이썬을 가리킬 수 있어 `No module named reinvent`
같은 오류가 납니다. 이 함수는 커널과 같은 파이썬(`sys.executable`)을 쓰고,
설정 파일이 실제로 있는지 먼저 확인합니다.

In [ ]:
import sys, subprocess

def reinvent(toml_path, log_path):
    """REINVENT 실행 — 셸 변수 치환에 기대지 않는다."""
    for nm, v in (("설정 파일", toml_path), ("로그 파일", log_path)):
        if not isinstance(v, str) or not v:
            raise ValueError(f"{nm} 경로가 비어 있습니다: {v!r} — 앞의 설정 셀을 먼저 실행하세요")
    if not os.path.exists(toml_path):
        raise FileNotFoundError(f"설정 파일이 없습니다: {toml_path}")
    cmd = [sys.executable, "-m", "reinvent.Reinvent", "-l", log_path, toml_path]
    print("실행:", toml_path)
    r = subprocess.run(cmd, cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError(f"REINVENT 실패 (코드 {r.returncode}) — 로그: {log_path}")
    print("완료. 로그:", log_path)

In [ ]:
import sys, subprocess

def reinvent(toml_path, log_path):
    """REINVENT 실행 — 셸 변수 치환에 기대지 않는다."""
    for nm, v in (("설정 파일", toml_path), ("로그 파일", log_path)):
        if not isinstance(v, str) or not v:
            raise ValueError(f"{nm} 경로가 비어 있습니다: {v!r} — 앞의 설정 셀을 먼저 실행하세요")
    if not os.path.exists(toml_path):
        raise FileNotFoundError(f"설정 파일이 없습니다: {toml_path}")
    cmd = [sys.executable, "-m", "reinvent.Reinvent", "-l", log_path, toml_path]
    print("실행:", toml_path)
    r = subprocess.run(cmd, cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError(f"REINVENT 실패 (코드 {r.returncode}) — 로그: {log_path}")
    print("완료. 로그:", log_path)

REINVENT의 파라미터를 이해하기 위해, 같은 reward 함수를 쓰되 sigma 값만 다른 두 모델을 학습시켜 봅니다. sigma는 prior 모델과 현재 학습 중인 모델 사이의 균형을 조절합니다. 다시 말해 reward 함수의 가중치(weight)이며, 값이 클수록 reward 함수가 학습에 더 큰 영향을 줍니다.

sigma를 더 뚜렷하게 보여주기 위해 다소 특이한 과제를 골랐습니다 — 분자량(MW)이 700~1200 Da인 큰 분자에 보상을 주는 것입니다. 이를 위해 두 개의 TOML 설정 파일을 만듭니다. 각 파일에는 global parameters, learning strategy, stage를 정의하며, sigma는 learning strategy에 들어갑니다.

먼저 저자들이 권장한 sigma = 128을 사용하고, 이어서 sigma = 16을 시도해 봅니다.

In [ ]:
# ── 이 실험에서 바꾸는 값은 이 두 개뿐입니다 ──────────────────
SIGMA = 128                      # 비교 실행에서는 16 으로 바꿉니다
TAG   = "sigma128_unusualmw"     # 비교 실행에서는 "sigma16_unusualmw"

# 직접 학습하면 로그는 "..._mine" 에 쌓입니다.
# 미리 받아 둔 비교용 로그(tb_logs_sigma128_unusualmw 등)를 덮어쓰지 않기 위해서입니다.
global_parameters = f"""
run_type = "staged_learning"
device = "{DEVICE}"
tb_logdir = "{wfile('sigma', f'tb_{TAG}_mine')}"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("/content/REINVENT4/priors/reinvent.prior")
agent_filename = prior_filename

parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "{wfile('sigma', f'stage_{TAG}')}"

batch_size = 64

use_checkpoint = false
"""


learning_strategy = f"""
[learning_strategy]

type = "dap"
sigma = {SIGMA}
rate = 0.0001
"""

stages = f"""
[[stage]]

max_score = 1.0
min_steps = 50
max_steps = 300

chkpt_file = '{wfile('sigma', f'stage_{TAG}.chkpt')}'

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]
[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]

[[stage.scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"  # 출력에 표시될 사용자 지정 이름
weight = 1  # 이 컴포넌트의 상대적 중요도를 조절하는 가중치

transform.type = "double_sigmoid"
transform.high = 1200.0
transform.low = 700.0
transform.coef_div = 1200.0
transform.coef_si = 20.0
transform.coef_se = 20.0
"""



In [ ]:
%cd /content/REINVENT4
config = global_parameters + parameters + learning_strategy + stages

toml_config_filename = wfile("sigma", f"stage{SIGMA}.toml")

with open(toml_config_filename, "w") as tf:
    tf.write(config)

print("작성:", toml_config_filename)
print("로그:", wfile("sigma", f"tb_{TAG}_mine_0"), " ← 직접 학습하면 여기에 쌓입니다")

# ── 비교 실행(sigma = 16)을 직접 해 보려면 ───────────────────────
#   위 셀에서  SIGMA = 16,  TAG = "sigma16_unusualmw"  로 바꾸고
#   위 셀 → 이 셀 → 학습 셀 순서로 다시 실행하면 됩니다.
#   파일 이름·로그 폴더·체크포인트·summary 접두사가 모두 TAG 를 따라가므로
#   앞서 만든 sigma=128 결과를 덮어쓰지 않습니다.


Colab CPU에서는 학습에 시간이 매우 오래 걸립니다. 그래서 위와 동일한 설정으로 우리가 미리 학습해 둔 chkpt 파일과 tensorboard 로그를 제공합니다. 따라서 다음 셀은 건너뛰고 tensorboard로 결과를 분석해도 됩니다. 직접 학습해 보고 싶다면 다음 셀을 실행하세요.

In [ ]:
%%time
# 이 셀은 선택 사항입니다. CPU 런타임에서는 매우 오래 걸립니다.
#   · 건너뛰어도 됩니다 — 아래 TensorBoard 비교는 미리 학습해 둔 로그로 진행됩니다.
#   · 진행 상황은 화면이 아니라 로그 파일에 쌓입니다 (-l 옵션).
#     다른 셀에서  !tail -20 {log}  로 확인하세요.
log = wfile("sigma", "stage.log")
reinvent(toml_config_filename, log)


모델의 학습 성능을 분석하기 위해 tensorboard를 사용합니다. 먼저 Colab에서 tensorboard를 불러온 뒤, 로그 폴더를 지정해 실행합니다.

In [ ]:
%load_ext tensorboard

결과를 살펴보고, 어떤 모델이 분자량 700~1200 범위의 분자를 생성하도록 학습되었는지 확인하세요.

In [ ]:
# 3절 sigma 비교용 TensorBoard 로그 (없으면 받아서 01_sigma 폴더에 놓습니다)
TB128, TB16, TB16N = ensure("sigma",
                            "tb_logs_sigma128_unusualmw",
                            "tb_logs_sigma16_unusualmw",
                            "tb_logs_sigma16_normalmw")
print(TB128)


In [ ]:
%tensorboard --logdir $TB128 --load_fast true


In [ ]:
%tensorboard --logdir $TB16 --load_fast true


tensorboard의 **First 30 Structures**를 보면, `sigma128` 모델이 `sigma16` 모델보다 훨씬 큰 분자를 생성하는 것을 알 수 있습니다. 또한 tensorboard의 **Molecular weight (raw)** 값이 `sigma128` 모델에서는 약 800 Da에 가까워지는 반면, `sigma16` 모델에서는 270 Da 부근에서 오르내립니다.

즉, 같은 분자(MW 700 ~ 1200)에 보상을 주더라도 두 모델의 학습 양상은 서로 매우 다릅니다. 그렇다면 MW 200~600 범위의 분자를 생성하려는 경우엔 sigma=16이 여전히 부족할까요? 확인해 봅시다! 이 설정으로 이미 학습해 둔 모델의 로그 파일을 사용합니다.

In [ ]:
%tensorboard --logdir $TB16N --load_fast true


스텝별 평균 점수(score)가 이전 모델보다 높습니다. 완벽하진 않지만 훨씬 낫습니다. reward 함수의 영향이 더 작은데도 더 높은 점수를 받는 셈인데, 목표 범위(MW 200~600)가 prior가 이미 잘 만들어 내는 영역이라 적은 힘으로도 도달한다고 읽는 것이 자연스럽습니다. REINVENT의 prior는 ChEMBL로 사전학습되어 있으므로 ChEMBL의 분자량 분포가 그 배경일 가능성이 큽니다 — 다만 이 노트북 안에서 그것까지 측정하지는 않았습니다. 확인하려면 prior에서 그대로 샘플링해 MW 분포를 그려 보면 됩니다.

따라서 sigma 파라미터는 학습에 큰 영향을 미칩니다 — 특히 분자를 생성하거나 일반적인 ChEMBL 분포를 벗어난 chemical space를 탐색하는 과제에서 그렇습니다. 다시 말해, reward 함수로 ChEMBL에서 오는 편향(bias)을 깨고 싶다면 더 큰 sigma 값이 필요할 수 있습니다.

큰 분자에 보상을 주는 것은 다소 극단적이었을 수 있습니다 — 대부분의 신약개발 프로젝트에서는 그렇게 큰 분자를 원하지 않기 때문입니다. 대신, 회전 결합 수(rotatable bonds)가 적은 분자에 보상을 줘 봅시다. 회전 결합이 적은 분자가 더 높은 점수를 받도록 하려면 TOML 파일을 어떻게 조정해야 할까요?

이 경우에는 raw 값을 reverse sigmoid 함수로 변환(transform)합니다 — 회전 결합이 적은 분자에 보상을 주고 싶기 때문입니다.

REINVENT4의 transform 종류는 다음과 같습니다:

**Sigmoid**: 임의의 실수 값을 (0, 1) 구간으로 부드럽게 변환합니다.

**Reverse_sigmoid**: sigmoid와 같지만 방향이 반대입니다. 낮은 입력값을 높은 출력으로(그 반대도) 매핑하고 싶을 때 사용합니다.

**Double_sigmoid**: 범위의 가운데 값을 강조하고 양 극단을 억제합니다. "중간" 입력을 부각하고 극단을 배제하고 싶을 때 사용합니다.

**Right_step**: 급격한 임계처리 — 임계값 미만은 모두 0, 초과는 1로 매핑합니다.

**Left_step**: 반대 임계처리 — 낮은 입력에 대해 출력을 높게(1) 두고 싶을 때 사용합니다.

**Step**: 특정 구간의 값만 1로, 나머지는 0으로 매핑합니다.

**Value_mapping**: 이산 값들 사이의 매핑 — 범주(category)에 수치를 부여합니다.

추가로, RL 과정에서 scaffold 다양성을 높이기 위해 diversity filter를 넣습니다. REINVENT4의 diversity filter는 특정 scaffold를 저장하는 "bucket" 메모리를 사용하며, scaffold bucket이 가득 차면 해당 분자에 0점을 부여합니다. 이는 분자 생성에서 mode collapse(다양성 붕괴)를 방지합니다.

In [ ]:
global_parameters = f"""
run_type = "staged_learning"
device = "{DEVICE}"
tb_logdir = "{wfile('rotbond', 'tb_numrotbond_mine')}"
json_out_config = "_stage1.json"
"""

prior_filename = os.path.join("/content/REINVENT4/priors/reinvent.prior")
agent_filename = prior_filename

parameters = f"""
[parameters]

prior_file = "{prior_filename}"
agent_file = "{agent_filename}"
summary_csv_prefix = "{wfile('rotbond', 'stage_numrotbond')}"

batch_size = 100

use_checkpoint = false
"""


learning_strategy = """
[learning_strategy]

type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]

type = "IdenticalMurckoScaffold" # IdenticalTopologicalScaffold,
                                 # ScaffoldSimilarity, PenalizeSameSmiles
bucket_size = 10                 # 메모리 크기 (화합물 개수 기준)
minscore = 0.4                   # 이 임계값을 넘어야만 메모리에 저장
minsimilarity = 0.4              # ScaffoldSimilarity의 최소 유사도
penalty_multiplier = 0.5         # PenalizeSameSmiles의 페널티 계수


"""

stage = f"""
[[stage]]

max_score = 1.0
min_steps = 50
max_steps = 300

chkpt_file = '{wfile('rotbond', 'numrotbond_reversesig.chkpt')}'

[stage.scoring]
type = "geometric_mean"

###### 원치 않는 구조를 걸러내기 위해 alerts는 유지

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"

params.smarts = [
    "[*;r8]",
    "[*;r9]",
    "[*;r10]",
    "[*;r11]",
    "[*;r12]",
    "[*;r13]",
    "[*;r14]",
    "[*;r15]",
    "[*;r16]",
    "[*;r17]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#7;!n]",
    "[#7;!n][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#16;!s]",
    "[#8;!o][C;!$(C(=[O,N])[N,O])][#8;!o]",
    "[#16;!s][C;!$(C(=[O,N])[N,O])][#16;!s]"
]


[[stage.scoring.component]]
[stage.scoring.component.NumRotBond]

[[stage.scoring.component.NumRotBond.endpoint]]
name = "ROTB"
weight = 1
transform.type = "reverse_sigmoid"
transform.high = 20
transform.low = 0
transform.k = 0.5

"""

In [ ]:
config = global_parameters + parameters + learning_strategy + stage
toml_config_filename = wfile("rotbond", "reward_numrotb.toml")
with open(toml_config_filename, 'w') as tf:
    tf.write(config)
print("작성:", toml_config_filename)


In [ ]:
# 모델 학습은 선택 사항이므로 이 셀은 건너뛰어도 됩니다
log = wfile("rotbond", "numrotb.log")
reinvent(toml_config_filename, log)


**참고:** RL 과정에서 tensorboard로 scaffold 반복(repetition)을 확인하는 것을 잊지 마세요.

In [ ]:
# 회전결합 모델의 학습 로그 — 위 학습 셀을 건너뛰었다면 받아옵니다
(TB_ROT,) = ensure("rotbond", "tb_logs_numrotbond")


In [ ]:
%tensorboard --logdir $TB_ROT --load_fast true


우리가 학습한 모델을 사용해 분자를 sampling하려면 어떻게 할까요? REINVENT용 `toml` 파일이 따로 필요합니다. 예시는 다음과 같습니다:

In [ ]:
global_sampling_parameters = f"""
run_type = "sampling"
device = "{DEVICE}"
json_out_config = "{wfile('rotbond', '_sampling.json')}"
"""

# 앞 절에서 쓴 `parameters` 를 덮어쓰지 않도록 이름을 따로 둡니다.
ROT_CHKPT = wfile("rotbond", "numrotbond_reversesig.chkpt")
ROT_SAMPLED = wfile("rotbond", "sampling_numrotbond.csv")

sampling_parameters = f"""
[parameters]
model_file = "{ROT_CHKPT}"
output_file = "{ROT_SAMPLED}"
num_smiles = 1200
unique_molecules = true
randomize_smiles = true
"""


In [ ]:
samp_conf = global_sampling_parameters + sampling_parameters
samp_toml = wfile("rotbond", "sampling_numrotbond.toml")
with open(samp_toml, "w") as tf:
    tf.write(samp_conf)
print("작성:", samp_toml)


In [ ]:
# sampling에는 학습된 체크포인트가 필요합니다.
# 위 학습 셀을 건너뛰었다면 미리 학습해 둔 체크포인트를 받아옵니다.
ensure("rotbond", "numrotbond_reversesig.chkpt")


In [ ]:
# sampling은 학습보다 훨씬 빠릅니다
log = wfile("rotbond", "sampling.log")
reinvent(samp_toml, log)


In [ ]:
# REINVENT가 생성한 데이터를 확인합니다
!head $ROT_SAMPLED


## 과제: 분자 점수 매기기 (Scoring)
이제 1200개의 분자를 생성했으니, 여러 scoring 함수로 평가해 다음을 파악할 수 있습니다:
- 최적화가 얼마나 잘 되었는지
- 다른 어떤 물성이 함께 영향을 받았는지
- 분자들이 drug-likeness를 유지하는지

### REINVENT4에서 사용 가능한 Scoring 함수

REINVENT4는 RDKit 연동을 통해 다양한 분자 descriptor를 제공합니다:

#### **Drug-Likeness & ADMET 물성**
| 함수 | 설명 |
|----------|-------------|
| **QED** | 정량적 drug-likeness |
| **SlogP** | 친유성 (Crippen) |
| **TPSA** | 위상학적 극성 표면적 (TPSA) |
| **HBondAcceptors** | 수소결합 수용체 수 |
| **HBondDonors** | 수소결합 공여체 수 |

#### **구조 descriptor**
| 함수 | 설명 | 활용 |
|----------|-------------|----------|
| **NumRotBond** | 회전 결합 수 | 유연성/강직성 |
| **GraphLength** | 최장 경로 길이 | 분자 크기 |
| **NumRings** | 전체 고리 수 | 복잡도 |
| **NumAromaticRings** | 방향족 고리 수 | π-π 상호작용 |
| **Csp3** | sp³ 탄소 비율 | 3차원적 특성 |

#### **입체화학 & 복잡도**
| 함수 | 설명 |
|----------|--------------|
| **NumAtomStereoCenters** | 카이랄 중심 수 |
| **NumHeavyAtoms** | 비수소 원자 수 |
| **NumHeteroAtoms** | 비탄소 원자 수 |

### 다중 물성 Scoring 전략

위의 여러 descriptor로 생성 분자에 점수를 매겨 종합적인 프로파일을 얻을 수 있습니다. scoring 전략을 담은 또 다른 `toml` 파일을 사용합니다.

In [ ]:
ROT_SCORED = wfile("rotbond", "scoring_numrotbond.csv")

global_scoring_parameters = f"""
run_type = "scoring"
json_out_config = "{wfile('rotbond', '_scoring.json')}"
"""

scoring_parameters = f"""
[parameters]
smiles_file = "{ROT_SAMPLED}"
output_csv = "{ROT_SCORED}"
"""
scoring = """
[scoring]
type = "geometric_mean"

[[scoring.component]]
[scoring.component.GraphLength]
[[scoring.component.GraphLength.endpoint]]
name = "GraphLength" # 최장 경로의 결합(bond) 수
weight = 0.2
transform.type = "reverse_sigmoid"
transform.high = 40
transform.low = 20
transform.k = 0.5

######
#  원하는 만큼 scoring 함수를 추가하세요. 각 함수의 weight를 정의하고 적절히 transform 하세요.
######





"""

In [ ]:
scoring_conf = global_scoring_parameters + scoring_parameters + scoring
score_toml = wfile("rotbond", "scoring_numrotbond.toml")
with open(score_toml, "w") as tf:
    tf.write(scoring_conf)
print("작성:", score_toml)


In [ ]:
log = wfile("rotbond", "scoring.log")
reinvent(score_toml, log)


분자를 점수와 함께 시각화할 차례입니다.

In [ ]:
# 점수가 매겨진 분자 로드

scored_mols = pd.read_csv(ROT_SCORED)

# 시각화를 위해 SMILES를 RDKit 분자 객체로 변환
scored_mols['mols'] = [Chem.MolFromSmiles(smi) for smi in scored_mols["SMILES"]]

# 점수 기준 정렬 (높은 분자 먼저)
scored_mols = scored_mols.sort_values('Score', ascending=False).reset_index(drop=True)

# 인터랙티브 분자 그리드
mols2grid.display(
    scored_mols,
    mol_col='mols',
    dpi=300,
    subset=['Score', 'GraphLength (raw)'],  # 분석하려면 scoring 컴포넌트를 추가할 수도 있습니다
    transform={
        "Score": lambda x: f"{x:.2f}", # 긴 raw 값 대신 변환된 값을 표시하려면 다른 컴포넌트도 추가
        "GraphLength (raw)": lambda x: f"{x:.2f}", # scoring 컴포넌트 예시
    },
    n_items_per_page=12,
    size=(200, 200),
)


---

# 💡 [예시로 풀어보는 과제] 알려진 저해제의 물성 프로파일을 목표로 한 RL

REINVENT4의 마지막 과제입니다. ChEMBL에서 가져온 M. tuberculosis TrxR 데이터셋(Assay ID: CHEMBL2395745)을 업로드해 두었습니다. 이 데이터셋에는 MW, LogP, HBA, HBD, PSA, ROTB, 방향족 고리 수, QED, 저해율(%) 값을 가진 13개의 분자가 있습니다.

활성값을 예측하는 별도 모델을 만드는 대신 이 13개 저해제가 공유하는 **물리화학적 물성 프로파일**(분자량·logP·TPSA·수소결합·회전결합)을 목표로 삼아, 그 범위에 맞는 새 분자를 생성하도록 RL을 돌립니다.

**흐름:** ① 13개 저해제의 물성 범위 파악 → ② 그 범위를 목표(reward)로 하는 RL → ③ 생성 분자 sampling·평가 → ④ 상위 10% 선별 + 저해제 물성과 분포 비교


In [ ]:
dataset = pd.read_csv('/content/REINVENT4/mt_trxr.csv')
dataset['Inh_value(%)'] = dataset['Value']
dataset = dataset.drop(columns=['Value'])
dataset['mol'] = [Chem.MolFromSmiles(smi) for smi in dataset["Smiles"]]
dataset

## ① 13개 저해제의 물성 프로파일 파악

위 셀에서 이미 `dataset`(mt_trxr.csv)을 불러왔습니다. 여기서는 13개 분자의 SMILES로부터 RDKit로 주요 물성을 계산해 **목표로 삼을 범위**를 구합니다.

In [ ]:
from rdkit.Chem import Descriptors
import numpy as np

def calc_props(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    return {"MW":    Descriptors.MolWt(m),
            "SlogP": Descriptors.MolLogP(m),
            "TPSA":  Descriptors.TPSA(m),
            "HBA":   Descriptors.NumHAcceptors(m),
            "HBD":   Descriptors.NumHDonors(m),
            "ROTB":  Descriptors.NumRotatableBonds(m)}

# 13개 저해제의 물성을 직접 계산해 ② 에서 목표로 삼을 범위를 확인한다
props  = pd.DataFrame([calc_props(s) for s in dataset["Smiles"]])
ranges = props.agg(["min", "max", "mean"]).T.round(1)
ranges.columns = ["min", "max", "mean"]
print("13개 저해제의 물성 범위 — 아래 [min, max] 가 ② 의 reward 목표가 된다")
display(ranges)


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(dataset['PSA'], dataset['Inh_value(%)'])
plt.xlabel('TPSA'); plt.ylabel('Inhibition (%)')
plt.title('13 inhibitors - TPSA vs inhibition')
plt.show()


In [ ]:
mols2grid.display(dataset, mol_col='mol', subset=['Inh_value(%)'], size=(100, 100))

## ② 물성 범위를 목표로 하는 RL 설정 만들기

각 물성이 저해제들의 `[min, max]` 범위 안에 들어오면 높은 점수를 주도록 **double_sigmoid** transform을 사용하거나 각 항목에 맞는 transform을 사용합니다. 여러 물성을 동시에 맞춰야 하므로 `geometric_mean`으로 합칩니다.

In [ ]:
prior_filename = "/content/REINVENT4/priors/reinvent.prior"

components = f"""
[[stage.scoring.component]]
[stage.scoring.component.QED]
[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 1

[[stage.scoring.component]]
[stage.scoring.component.TPSA]
[[stage.scoring.component.TPSA.endpoint]]
name = "TPSA"
weight = 0.5
transform.type = "reverse_sigmoid"
transform.high = 120
transform.low = 40
transform.k = 0.3
"""


config = f"""run_type = "staged_learning"
device = "{DEVICE}"
tb_logdir = "{wfile('trxr', 'tb_trxr_mine')}"
json_out_config = "{wfile('trxr', '_trxr.json')}"

[parameters]
prior_file = "{prior_filename}"
agent_file = "{prior_filename}"
summary_csv_prefix = "{wfile('trxr', 'trxr_stage')}"
batch_size = 64
use_checkpoint = false

[learning_strategy]
type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]
type = "IdenticalMurckoScaffold"
bucket_size = 25
minscore = 0.4

[[stage]]
chkpt_file = "{wfile('trxr', 'trxr_stage.chkpt')}"
termination = "simple"
max_score = 0.8
min_steps = 10
max_steps = 30

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]
[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"
params.smarts = ["[*;r{{8-17}}]", "[#8][#8]", "[#6;+]", "[#16][#16]", "C#C"]

{components}"""

TRXR_TOML = wfile("trxr", "trxr_staged.toml")
with open(TRXR_TOML, "w") as f:
    f.write(config)
print("작성:", TRXR_TOML)


## ③ RL 실행 & 학습곡선

> ⏳ **주의:** Colab CPU에서는 시간이 걸립니다(수십 분). GPU 런타임이면 훨씬 빠릅니다. 시간이 부족하면 `max_steps`를 더 줄이거나 이 셀을 건너뛰세요.

In [ ]:
%cd /content/REINVENT4
log = wfile("trxr", "trxr.log")
reinvent(TRXR_TOML, log)


In [ ]:
# TrxR RL 학습 로그 — 학습 셀을 건너뛰었다면 받아옵니다
(TB_TRXR,) = ensure("trxr", "tb_trxr_0")


In [ ]:
# 학습곡선 — 스텝이 지날수록 평균 점수가 오르면 물성 목표에 수렴하는 중
%tensorboard --logdir $TB_TRXR --load_fast true


## ④ 생성 분자 sampling → 상위 10% 선별 → 저해제와 비교

학습한 agent(checkpoint)에서 분자를 sampling한 뒤, 각 분자가 목표 물성 범위에 얼마나 들어맞는지로 점수를 매겨 **상위 10%**를 선별합니다. 그리고 생성 분자의 물성 분포가 실제 13개 저해제와 얼마나 겹치는지 확인합니다.

In [ ]:
# sampling에는 학습된 체크포인트가 필요합니다.
# 위 RL 셀을 건너뛰었거나 시간이 부족했다면 미리 학습해 둔 체크포인트를 받아옵니다.
(TRXR_CHKPT,) = ensure("trxr", "trxr_stage.chkpt")


In [ ]:
# 학습한 agent에서 분자 생성
TRXR_GEN = wfile("trxr", "trxr_generated.csv")

sampling_toml = f"""
run_type = "sampling"
device = "{DEVICE}"

[parameters]
model_file = "{TRXR_CHKPT}"
output_file = "{TRXR_GEN}"
num_smiles = 1000
unique_molecules = true
randomize_smiles = true
"""
samp_toml = wfile("trxr", "sampling_trxr.toml")
with open(samp_toml, "w") as f:
    f.write(sampling_toml)

log = wfile("trxr", "sampling_trxr.log")
reinvent(samp_toml, log)


In [ ]:
# 생성 분자 물성 계산 + "목표 범위 적합도(match)" 점수 → 상위 10%

TRXR_SCORED = wfile("trxr", "trxr_scoring.csv")

global_scoring_parameters = f"""
run_type = "scoring"
json_out_config = "{wfile('trxr', '_scoring.json')}"
"""

trxr_scoring_parameters = f"""
[parameters]
smiles_file = "{TRXR_GEN}"
output_csv = "{TRXR_SCORED}"
"""
scoring = """
[scoring]
type = "geometric_mean"

[[scoring.component]]
[scoring.component.GraphLength]
[[scoring.component.GraphLength.endpoint]]
name = "GraphLength" # 최장 경로의 결합(bond) 수
weight = 0.2
transform.type = "reverse_sigmoid"
transform.high = 40
transform.low = 20
transform.k = 0.5

[[scoring.component]]
[scoring.component.QED]
[[scoring.component.QED.endpoint]]
name = "QED"  # 정량적 drug-likeness
weight = 0.2

[[scoring.component]]
[scoring.component.MolecularWeight]
[[scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"  # user chosen name for output
weight = 0.2  # weight to fine-tune the relevance of this component

# A transform ensures that the output from the scoring component ranges
# from 0 to 1 to serve as a proper score.  Here we use a double sigmoid
# to transform weights into the range 200-500 a.u.
transform.type = "double_sigmoid"
transform.high = 500.0
transform.low = 200.0
transform.coef_div = 500.0
transform.coef_si = 20.0
transform.coef_se = 20.0

[[scoring.component]]
[scoring.component.TPSA]
[[scoring.component.TPSA.endpoint]]
name = "TPSA"
weight = 0.2
transform.type = "double_sigmoid"
transform.high = 140.0
transform.low = 0.0
transform.coef_div = 140.0
transform.coef_si = 20.0
transform.coef_se = 20.0



"""
scoring_conf = global_scoring_parameters + trxr_scoring_parameters + scoring
TRXR_SCORE_TOML = wfile("trxr", "trxr_scoring.toml")
with open(TRXR_SCORE_TOML, "w") as tf:
    tf.write(scoring_conf)
print("작성:", TRXR_SCORE_TOML)


In [ ]:
log = wfile("trxr", "scoring_trxr.log")
reinvent(TRXR_SCORE_TOML, log)


In [ ]:
# 앞의 sampling·scoring을 건너뛰었다면 미리 만들어 둔 결과 파일을 받아옵니다
TRXR_GEN, TRXR_SCORED = ensure("trxr", "trxr_generated.csv", "trxr_scoring.csv")


In [ ]:
# 생성 분자 물성 계산 + "목표 범위 적합도(match)" 점수 → 상위 10%
gen = pd.read_csv(TRXR_SCORED)

top10 = gen.sort_values("Score", ascending=False).head(int(len(gen) * 0.10)).reset_index(drop=True)
print(f"생성 {len(gen)}개 → 상위 10% = {len(top10)}개 (평균 적합도 {top10['Score'].mean():.2f})")
top10.head()

In [ ]:
# 생성 분자 vs 13개 저해제 — 물성 분포 비교
# (Colab 기본 matplotlib에는 한글 폰트가 없어 그래프 글자는 영문으로 둡니다)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols = [("Molecular weight (raw)", "MW"), ("QED (raw)", "QED"), ("TPSA (raw)", "PSA")]
for ax, (col, ref_col) in zip(axes.ravel(), cols):
    ax.hist(gen[col],   bins=40, density=True, alpha=0.35, color="green", label="Generated (all)")
    ax.hist(top10[col], bins=40, density=True, alpha=0.55, color="navy",  label="Top 10%")
    ax.hist(dataset[ref_col], bins=40, density=True, alpha=0.45, color="red", label="Inhibitors (13)")
    ax.set_title(col); ax.legend(fontsize=8)
plt.suptitle("Property distributions - generated vs top 10% vs known inhibitors")
plt.tight_layout(); plt.show()


In [ ]:
# 상위 10% 분자 구조 보기
import mols2grid
top10 = top10.copy()
top10["mol"] = [Chem.MolFromSmiles(s) for s in top10["SMILES"]]
mols2grid.display(top10, mol_col="mol",
                  subset=["Score", "Molecular weight (raw)", "QED (raw)"],
                  transform={"Score": lambda x: f"{x:.2f}",
                             "Molecular weight (raw)":    lambda x: f"{x:.0f}",
                             "QED (raw)": lambda x: f"{x:.1f}"},
                  n_items_per_page=12, size=(200, 200))

## ⑤ 다양성 — 점수만 보면 놓치는 것

강의에서 본 대로 목표 지향 최적화는 점수를 올리는 대신 다양성을 깎습니다.
평균 유사도 하나로 요약하는 **IntDiv** 과, 서로 충분히 떨어진 덩어리를 세는 **#Circles** 를
함께 봅니다. 둘 다 같은 fingerprint·Tanimoto 행렬에서 나오지만 요약 방식이 다릅니다 —
IntDiv 은 평균이고, #Circles 는 개수입니다.


In [ ]:
# 생성 분자의 다양성 — IntDiv 과 #Circles
import random
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

_gen_fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def _fps(smiles):
    out = []
    for s in smiles:
        m = Chem.MolFromSmiles(str(s))
        if m is not None:
            out.append(_gen_fp.GetFingerprint(m))
    return out

def int_div(smiles):
    """내부 다양성 = 1 - 쌍별 Tanimoto 평균"""
    fps = _fps(smiles)
    tot = n = 0
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        tot += sum(sims); n += len(sims)
    return 1 - tot / n if n else float("nan")

def n_circles(smiles, sim_threshold=0.65, n_restarts=3, seed=0):
    """서로 sim_threshold 미만으로만 닮은 분자를 최대 몇 개 고를 수 있는가 (탐욕 근사)"""
    fps = _fps(smiles)
    best = 0
    for r in range(n_restarts):
        order = list(range(len(fps)))
        random.Random(seed + r).shuffle(order)
        centers = []
        for i in order:
            if centers and max(DataStructs.BulkTanimotoSimilarity(fps[i], centers)) >= sim_threshold:
                continue
            centers.append(fps[i])
        best = max(best, len(centers))
    return best

for label, df in (("생성 전체", gen), ("상위 10%", top10)):
    smi = df["SMILES"]
    print(f"{label:8s} n={len(smi):5d}   IntDiv {int_div(smi):.3f}   "
          f"#Circles {n_circles(smi)}  (Tanimoto 0.65 기준)")

print("\n점수가 높은 쪽일수록 #Circles 가 줄어들면, 최적화가 다양성을 깎은 것이다.")


### 🎯 이제 여러분 차례

이 예시는 **물성 프로파일 매칭**으로 접근했습니다. 다음을 바꿔가며 직접 실험해 보세요:

- **목표 물성을 조정** — QED·방향족 고리 수(`NumAromaticRings`) 등을 추가하거나, 범위를 좁혀 더 까다롭게 만들기
- **Transfer Learning으로 확장** — 위에서 선별한 상위 10% 분자(또는 13개 저해제)로 모델을 fine-tune해 타깃 화학공간에 더 초점 맞추기 (과제 원문의 마지막 단계, 상단 transfer_learning 예시 참고)


> 정답은 하나가 아닙니다. reward 컴포넌트·weight·transform 범위를 바꿔가며 생성 분자가 어떻게 달라지는지 관찰하는 것이 이 실습의 핵심입니다.


---
---

# 🧩 이어지는 실습 — LibInvent로 R-group 설계하기

여기까지는 **아무것도 주지 않고** 분자를 통째로 만들었습니다. 실무에서 더 자주 만나는 문제는
그 반대입니다 — **골격은 이미 정해져 있고, 거기에 무엇을 붙일지**를 찾는 일입니다.
이것을 scaffold decoration(골격 장식)이라 하고, REINVENT4에서는 **LibInvent** 모델이 맡습니다.

| | de novo (앞 실습) | **LibInvent** |
|---|---|---|
| prior | `reinvent.prior` | `libinvent.prior` |
| 주는 것 | 없음 | **골격 + 부착점** |
| 만드는 것 | 분자 전체 | **부착점에 붙는 R-group** |
| 출력 컬럼 | SMILES, NLL | SMILES, **Scaffold**, **R-groups**, NLL |

앞에서 설치·경로 설정을 이미 끝냈으므로 그대로 이어서 진행합니다.

In [ ]:
# LibInvent 실습 경로 — 앞의 wfile 헬퍼를 그대로 씁니다
LI = lambda name: wfile("libinvent", name)

PRIOR = os.path.join(ROOT, "priors", "libinvent.prior")

print("산출물 폴더    :", wdir("libinvent"))
print("LibInvent prior:", PRIOR, "—", "있음" if os.path.exists(PRIOR) else "없음(경로 확인 필요)")


In [ ]:
# ── 이번 실습의 골격 : 4-anilinoquinazoline (EGFR 저해제 계열의 공통 골격)
#    퀴나졸린 6번 위치에 부착점 하나를 둡니다 — erlotinib·gefitinib이 측쇄를 다는 자리입니다.
SCAFFOLD = "[*]c1ccc2ncnc(Nc3ccccc3)c2c1"

m = Chem.MolFromSmiles(SCAFFOLD)
assert m is not None, "골격 SMILES를 읽지 못했습니다"
print("더미 원자(부착점) 개수 :", sum(1 for a in m.GetAtoms() if a.GetAtomicNum() == 0))

scaffold_file = LI("scaffolds.smi")
with open(scaffold_file, "w") as f:
    f.write(SCAFFOLD + "\n")          # 한 줄에 골격 하나
print("작성 :", scaffold_file)

Draw.MolToImage(m, size=(420, 260))


참고로 완성된 약물은 이 골격에 측쇄가 붙은 모습입니다. 아래 두 분자를 나란히 보면
**골격은 같고 R-group만 다르다**는 것이 한눈에 보입니다 — LibInvent가 하는 일이 바로
이 자리를 채우는 것입니다.

In [ ]:
ref = {"erlotinib": "C#Cc1cccc(Nc2ncnc3cc(OCC)c(OCCOC)cc23)c1",
       "gefitinib":  "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1"}
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in ref.values()],
                     legends=list(ref), molsPerRow=2, subImgSize=(330, 230))


---
## 3. 먼저 prior로 바로 생성해 보기

학습 없이 **LibInvent prior 그대로** R-group을 붙여 봅니다. 몇 초면 끝나므로,
골격 파일과 부착점 표기가 맞는지 확인하는 용도로도 좋습니다.

`num_smiles` 는 **입력 골격 하나당 생성 개수**입니다 (전체 개수가 아닙니다).

In [ ]:
PRIOR_SAMPLED = LI("prior_sampled.csv")

sampling_toml = f"""
run_type = "sampling"
device = "{DEVICE}"

[parameters]
model_file = "{PRIOR}"
smiles_file = "{scaffold_file}"   # 골격 1줄 = 1개
output_file = "{PRIOR_SAMPLED}"
num_smiles = 100                  # 골격 하나당 100개
unique_molecules = true
randomize_smiles = false
"""
toml_path = LI("sampling_prior.toml")
with open(toml_path, "w") as f:
    f.write(sampling_toml)

log = LI("sampling_prior.log")
reinvent(toml_path, log)


In [ ]:
prior_df = pd.read_csv(PRIOR_SAMPLED)
print(prior_df.shape)
print("컬럼 :", list(prior_df.columns))
prior_df.head(10)


`Scaffold` 는 입력한 골격 그대로이고, `R-groups` 가 모델이 새로 만들어 낸 부분입니다.
`SMILES` 는 둘을 이어 붙인 완성 분자입니다.

**여기서 오류가 난다면** 부착점 표기 문제일 가능성이 큽니다. 위 골격 셀에서 `[*]` 를
`*` 로 바꿔 다시 만들어 보세요 — REINVENT 버전에 따라 받아들이는 표기가 다를 수 있습니다.

In [ ]:
# 붙은 R-group이 실제로 어떤 것들인지 세어 봅니다
rg = prior_df["R-groups"].value_counts()
print("서로 다른 R-group :", rg.size, "종")
rg.head(15)


---
## 4. 원하는 쪽으로 R-group을 몰아가기 (RL)

prior는 "그럴듯한" R-group을 붙일 뿐 목표가 없습니다. 앞 실습과 똑같은 방식으로
reward를 걸어 원하는 성질 쪽으로 끌어당깁니다. 달라지는 것은 두 줄뿐입니다 —
`prior_file` 이 `libinvent.prior` 이고, 골격 파일을 `smiles_file` 로 준다는 것.

여기서는 **완성 분자가 drug-like 범위에 들어오도록** 겁니다.

| 컴포넌트 | 의도 |
|---|---|
| QED | 전반적인 drug-likeness |
| MolecularWeight | 350~500 Da (double_sigmoid) |
| SlogP | 1~4 (double_sigmoid) |
| custom_alerts | 원치 않는 구조 배제 |

합치는 방식은 `geometric_mean` 입니다 — 어느 하나라도 0에 가까우면 전체가 내려가므로,
"모든 항목이 하한을 넘어야 한다"는 요구에 맞습니다.

In [ ]:
AGENT_CHKPT = LI("libinvent_druglike.chkpt")
TB_DIR      = LI("tb_libinvent")

rl_toml = f"""
run_type = "staged_learning"
device = "{DEVICE}"
tb_logdir = "{TB_DIR}"
json_out_config = "{LI('_libinvent.json')}"

[parameters]
prior_file = "{PRIOR}"
agent_file = "{PRIOR}"
smiles_file = "{scaffold_file}"
summary_csv_prefix = "{LI('libinvent_stage')}"
batch_size = 64
use_checkpoint = false

[learning_strategy]
type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]
type = "IdenticalMurckoScaffold"
bucket_size = 25
minscore = 0.4

[[stage]]
chkpt_file = "{AGENT_CHKPT}"
termination = "simple"
max_score = 0.8
min_steps = 10
max_steps = 50

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]
[[stage.scoring.component.custom_alerts.endpoint]]
name = "Alerts"
params.smarts = ["[*;r{{8-17}}]", "[#8][#8]", "[#6;+]", "[#16][#16]", "C#C"]

[[stage.scoring.component]]
[stage.scoring.component.QED]
[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 1

[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]
[[stage.scoring.component.MolecularWeight.endpoint]]
name = "MW"
weight = 1
transform.type = "double_sigmoid"
transform.high = 500.0
transform.low = 350.0
transform.coef_div = 500.0
transform.coef_si = 20.0
transform.coef_se = 20.0

[[stage.scoring.component]]
[stage.scoring.component.SlogP]
[[stage.scoring.component.SlogP.endpoint]]
name = "SlogP"
weight = 1
transform.type = "double_sigmoid"
transform.high = 4.0
transform.low = 1.0
transform.coef_div = 4.0
transform.coef_si = 20.0
transform.coef_se = 20.0
"""

RL_TOML = LI("libinvent_rl.toml")
with open(RL_TOML, "w") as f:
    f.write(rl_toml)
print("작성 :", RL_TOML)


> ⏳ 아래 학습 셀은 **선택 사항**입니다. `max_steps = 50` 으로 짧게 잡아 두었지만
> CPU에서는 그래도 수 분 이상 걸립니다. 진행 상황은 화면이 아니라 로그 파일에 쌓입니다.

In [ ]:
%%time
log = LI("libinvent_rl.log")
reinvent(RL_TOML, log)


In [ ]:
# 학습이 도는 중인지 확인 (다른 셀에서 실행)
log = LI("libinvent_rl.log")
!tail -15 $log


In [ ]:
%load_ext tensorboard


In [ ]:
# 학습 곡선 — 스테이지 번호 _0 이 붙습니다
TB0 = TB_DIR + "_0"
if not os.path.exists(TB0):
    (TB0,) = ensure("libinvent", "tb_libinvent_0")
%tensorboard --logdir $TB0 --load_fast true


---
## 5. 학습한 agent로 다시 생성 → prior와 비교

같은 골격에 대해 **prior가 붙인 R-group**과 **학습한 agent가 붙인 R-group**이
어떻게 달라졌는지 보는 것이 이 실습의 핵심입니다.

In [ ]:
AGENT_SAMPLED = LI("agent_sampled.csv")

if os.path.exists(AGENT_CHKPT):
    # 직접 학습했다면 그 체크포인트로 생성한다
    sampling_toml = f"""
run_type = "sampling"
device = "{DEVICE}"

[parameters]
model_file = "{AGENT_CHKPT}"
smiles_file = "{scaffold_file}"
output_file = "{AGENT_SAMPLED}"
num_smiles = 100
unique_molecules = true
randomize_smiles = false
"""
    toml_path = LI("sampling_agent.toml")
    with open(toml_path, "w") as f:
        f.write(sampling_toml)

    log = LI("sampling_agent.log")
    reinvent(toml_path, log)
else:
    # 학습을 건너뛰었다면 미리 만들어 둔 생성 결과를 받아온다
    # (체크포인트는 90MB가 넘으므로 결과 CSV 만 배포합니다)
    print("학습된 체크포인트가 없어 미리 생성해 둔 결과를 받아옵니다.")
    (AGENT_SAMPLED,) = ensure("libinvent", "agent_sampled.csv")


In [ ]:
import matplotlib.pyplot as plt

agent_df = pd.read_csv(AGENT_SAMPLED)

def props(df):
    out = []
    for s in df["SMILES"]:
        mm = Chem.MolFromSmiles(str(s))
        if mm is None:
            continue
        out.append({"MW": Descriptors.MolWt(mm),
                    "SlogP": Descriptors.MolLogP(mm),
                    "QED": Descriptors.qed(mm)})
    return pd.DataFrame(out)

p_prior, p_agent = props(prior_df), props(agent_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
targets = {"MW": (350, 500), "SlogP": (1, 4), "QED": None}
for ax, col in zip(axes, ["MW", "SlogP", "QED"]):
    ax.hist(p_prior[col], bins=30, alpha=0.5, density=True, color="gray", label="prior")
    ax.hist(p_agent[col], bins=30, alpha=0.6, density=True, color="navy", label="RL agent")
    if targets[col]:
        ax.axvspan(*targets[col], color="orange", alpha=0.15, label="target range")
    ax.set_title(col); ax.legend(fontsize=8)
plt.suptitle("Same scaffold - R-groups before and after RL")
plt.tight_layout(); plt.show()

print(p_prior.describe().loc[["mean", "50%"]].round(2))
print(p_agent.describe().loc[["mean", "50%"]].round(2))


In [ ]:
# R-group 자체를 비교한다 — 무엇이 선택되었나
cmp = pd.DataFrame({
    "prior": prior_df["R-groups"].value_counts().head(10),
    "agent": agent_df["R-groups"].value_counts().head(10),
})
cmp


In [ ]:
# 학습된 agent가 만든 분자 보기
agent_df = agent_df.copy()
agent_df["mol"] = [Chem.MolFromSmiles(s) for s in agent_df["SMILES"]]
agent_df["MW"] = [round(Descriptors.MolWt(m), 1) if m else None for m in agent_df["mol"]]
mols2grid.display(agent_df.dropna(subset=["mol"]).head(24),
                  mol_col="mol", subset=["R-groups", "MW"],
                  n_items_per_page=12, size=(220, 200))


---
## 6. 🎯 이제 여러분 차례

**① 부착점을 두 개로 늘려 보기**

```python
SCAFFOLD = "[*]c1ccc2ncnc(Nc3cccc([*])c3)c2c1"
```

퀴나졸린 쪽과 아닐린 쪽에 각각 부착점을 둔 골격입니다. 출력의 `R-groups` 컬럼이
어떻게 달라지는지 보세요. 탐색 공간이 넓어지는 만큼 학습도 더 오래 걸립니다.

**② reward를 바꿔 보기**

- TPSA를 추가해 극성 표면적을 제한해 보세요 (`transform.type = "reverse_sigmoid"`)
- `NumRotBond` 를 넣어 유연성을 억제해 보세요
- MW 목표 범위를 좁혀(`400~450`) 더 까다롭게 만들어 보세요

**③ 다양성 보기**

앞 실습에서 쓴 IntDiv·#Circles를 여기에도 적용해, R-group이 몇 종류로 수렴하는지
세어 보세요. `bucket_size` 를 바꾸면 어떻게 달라지나요?

**④ 자기 과제의 골격으로 바꾸기**

실제로 담당하시는 표적의 hit 구조에서 바꾸고 싶지 않은 부분만 남기고 나머지를 `[*]`로
비우면, 그대로 이 노트북에 넣을 수 있습니다. 그것이 LibInvent의 실무 사용법입니다.

> 생성 결과가 좋아 보여도 **합성 가능성은 별개**입니다. 실제 과제에서는 SA score나
> 반응 기반 필터를 reward에 함께 넣습니다.


### 📁 이번 실습에서 만들어진 파일


In [ ]:
!find /content/REINVENT4/work -maxdepth 2 | sort
